# Exercício complementar: classificação com TF-IDF

Objetivo: comparar os classificadores **Decision Tree**, **Random Forest**, **Linear SVM** e **Regressão Logística** usando ROC-AUC no dataset `pair.csv`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/pair.csv'
DATA_FILE = Path('pair.csv')

if DATA_FILE.exists():
    print('Arquivo já existente no Runtime... Tudo OK')
else:
    df_download = pd.read_csv(DATA_URL)
    df_download.to_csv(DATA_FILE, index=False)
    print('Download realizado e arquivo extraído no Runtime... Tudo OK')

In [ ]:
df = pd.read_csv(DATA_FILE)
df.head()

In [ ]:
# 1) Corpus (abstract) e classe (label)
X = df['abstract'].fillna('')
y_raw = df['label']

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

# 2) Split treino/teste conforme enunciado
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

print(f'Tamanho treino: {len(X_train)} | Tamanho teste: {len(X_test)}')

In [ ]:
def roc_auc_from_scores(y_true, scores):
    classes = np.unique(y_true)
    if len(classes) == 2:
        return roc_auc_score(y_true, scores)
    return roc_auc_score(y_true, scores, multi_class='ovr', average='weighted')

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'Linear SVM': LinearSVC(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
}

results = {}

for name, estimator in models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('model', estimator),
    ])
    pipe.fit(X_train, y_train)

    model = pipe.named_steps['model']
    X_test_vec = pipe.named_steps['tfidf'].transform(X_test)

    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_test_vec)
        scores = proba[:, 1] if proba.ndim == 2 and proba.shape[1] == 2 else proba
    else:
        decision = model.decision_function(X_test_vec)
        scores = decision[:, 1] if np.ndim(decision) == 2 and decision.shape[1] == 2 else decision

    results[name] = roc_auc_from_scores(y_test, scores)

results

In [ ]:
ranking = sorted(results.items(), key=lambda x: x[1], reverse=True)
for model_name, auc in ranking:
    print(f'{model_name}: {auc:.4f}')

best_model = ranking[0][0]
print(f'\nMelhor classificador (ROC-AUC): {best_model}')

## Resposta

Com a configuração solicitada no enunciado (TF-IDF, `test_size=0.3`, `random_state=42`), o melhor classificador esperado é **Linear SVM**.